# Bootstrap 假设检验：两样本均值比较（CHFS 男女收入）

> 对应代码：`MathStatsCode/code_in_notes/Chap.15/bootstrap_twosample.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**实验目的**：检验男性收入是否显著高于女性，即 $H_0:\mu_1\leq\mu_2$（下标 1 为男性、2 为女性）。用 bootstrap 模拟原假设下学生氏化两样本 $t$ 统计量的分布，计算 bootstrap 临界值与 $p$ 值，并与渐近正态近似的结果比较。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.15`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致。


In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.15")

import stata_setup
stata_setup.config("/opt/Stata", "mp")



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


载入 2017 年 CHFS 个人数据，并用 `set seed` 固定随机种子。变量 `a3136` 为劳动收入，`a2003` 为性别（1 为男性、2 为女性）。


In [2]:
%%stata
clear all
use ../datasets/chfs2017_ind.dta, clear
set seed 8855



. clear all

. use ../datasets/chfs2017_ind.dta, clear
(chfs2017 ind. data, add demograhic info)

. set seed 8855

. 


**计算观测样本的 t 统计量**

`ttest a3136, by(a2003) unequal` 进行不等方差的两样本 t 检验。返回值 `r(mu_1)、r(mu_2)` 为两组均值，`r(N_1)、r(N_2)` 为样本量，`r(sd_1)、r(sd_2)` 为两组标准差，`r(t)` 为 t 统计量。我们把均值差 `diff_obs` 与 t 统计量 `t_obs` 保存为标量（scalar 在所有 Stata 会话中全局可见，便于后面的 `program` 与命令引用；与 local 不同，它不受程序作用域限制）。观测样本中 $t=14.13$，远大于渐近正态临界值。


In [3]:
%%stata
ttest a3136, by(a2003) unequal
scalar diff_obs = r(mu_1)-r(mu_2)
scalar t_obs = r(t)
di "diff = " diff_obs
di "t = " t_obs



. ttest a3136, by(a2003) unequal

Two-sample t test with unequal variances
------------------------------------------------------------------------------
   Group |     Obs        Mean    Std. err.   Std. dev.   [95% conf. interval]
---------+--------------------------------------------------------------------
       1 |  20,460     41921.1     325.055    46495.36    41283.96    42558.23
       2 |  13,906     34218.8    437.4588    51586.76    33361.32    35076.27
---------+--------------------------------------------------------------------
Combined |  34,366    38804.41    263.0579    48765.86     38288.8    39320.01
---------+--------------------------------------------------------------------
    diff |              7702.3    545.0054                6634.062    8770.538
------------------------------------------------------------------------------
    diff = mean(1) - mean(2)                                      t =  14.1325
H0: diff = 0                     Satterthwaite's degree

**bootstrap 重中心化的学生氏化统计量**

为了在 bootstrap 中模拟原假设成立时的分布，需要让"bootstrap 世界"中两组均值相等。做法是对学生氏化统计量**重中心化**：

$$
T^{*}=\frac{(\bar{x}_1^{*}-\bar{x}_2^{*})-(\bar{x}_1-\bar{x}_2)}{\sqrt{\frac{s_1^{*2}}{N_1}+\frac{s_2^{*2}}{N_2}}},
$$

即把每次再抽样得到的均值差减去观测样本的均值差（它在 bootstrap 世界中充当"真值"）。这样 $T^{*}$ 在 bootstrap 世界中的期望为 0，相当于原假设成立。

程序 `boot_twosample` 在每次再抽样后重新运行 `ttest`，并用 `return scalar` 把重中心化的统计量返回给 `bootstrap` 命令；`bootstrap tstar=r(tstar), reps(1000) saving(...)` 进行 1000 次重复，结果保存到 `bootstrap_twosample.dta` 备用。


In [4]:
%%stata
capture program drop boot_twosample
program define boot_twosample, rclass
    ttest a3136, by(a2003) unequal
    return scalar tstar = ((r(mu_1)-r(mu_2))-diff_obs)/sqrt(r(sd_1)^2/r(N_1)+r(sd_2)^2/r(N_2))
end
bootstrap tstar=r(tstar), reps(1000) saving(bootstrap_twosample.dta, replace): boot_twosample



. capture program drop boot_twosample

. program define boot_twosample, rclass
  1.     ttest a3136, by(a2003) unequal
  2.     return scalar tstar = ((r(mu_1)-r(mu_2))-diff_obs)/sqrt(r(sd_1)^2/r(N_
> 1)+r(sd_2)^2/r(N_2))
  3. end

. bootstrap tstar=r(tstar), reps(1000) saving(bootstrap_twosample.dta, replace)
> : boot_twosample
(running boot_twosample on estimation sample)

         excluded from the resampling because of missing values or other
         reasons. To exclude observations, press Break, save the data, drop
         any observations that are to be excluded, and rerun bootstrap.


(file bootstrap_twosample.dta not found)

Bootstrap replications (1,000): 

.

.

.

...

.

.

1,000 done

Bootstrap results                                      Number of obs = 127,012
                                                       Replications  =   1,000

      Command: boot_twosample
        tstar: r(tstar)



------------------------------------------------------------------------------
             |   Observed   Bootstrap                         Normal-based
             | coefficient  std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
       tstar |

          0   1.062685     0.00   1.000    -2.082825    2.082825
------------------------------------------------------------------------------

. 


**载入 bootstrap 结果**

`preserve` 暂存当前数据，`use bootstrap_twosample.dta` 载入 1000 个 $T^{*}$（变量名即 `bootstrap` 命令中指定的 `tstar`）。


In [5]:
%%stata
preserve
use bootstrap_twosample.dta, clear



. preserve

. use bootstrap_twosample.dta, clear
(bootstrap: boot_twosample)

. 


**右侧检验的临界值与 p 值**

`_pctile tstar, p(95)` 计算 $\{T_b^{*}\}$ 的 95% 分位数，即 5% 显著性水平下右侧检验（$H_1:\mu_1>\mu_2$）的 bootstrap 临界值；右侧 bootstrap $p$ 值即 $T^{*}>t$ 的比例。


In [6]:
%%stata
_pctile tstar, p(95)
di "右侧检验临界值 = " r(r1)
count if tstar>t_obs
di "右侧检验p值 = " r(N)/_N



. _pctile tstar, p(95)

. di "右侧检验临界值 = " r(r1)
右侧检验临界值 = 1.911602

. count if tstar>t_obs
  0

. di "右侧检验p值 = " r(N)/_N
右侧检验p值 = 0

. 


**双侧检验的临界值与 p 值**

双侧检验的 bootstrap 临界值为 $\{T_b^{*}\}$ 的 2.5% 与 97.5% 分位数；双侧（对称）$p$ 值即 $|T^{*}|>|t|$ 的比例。可以看到，bootstrap 双侧临界值 $[-1.746,2.193]$ 比渐近正态的 $[-1.96,1.96]$ 呈现出轻微的左偏——这反映了右偏的收入分布下学生氏化 t 统计量抽样分布的左偏性（样本均值本身仍是右偏的）。由于观测到的 $t=14.13$ 远大于临界值，两个 $p$ 值均为 0，可以在 1% 的显著性水平下拒绝原假设，即男性收入显著高于女性。


In [7]:
%%stata
_pctile tstar, p(2.5 97.5)
di "双侧检验临界值 = [" r(r1) ", " r(r2) "]"
count if abs(tstar)>abs(t_obs)
di "双侧检验p值 = " r(N)/_N



. _pctile tstar, p(2.5 97.5)

. di "双侧检验临界值 = [" r(r1) ", " r(r2) "]"
双侧检验临界值 = [-1.745559, 2.1934497]

. count if abs(tstar)>abs(t_obs)
  0

. di "双侧检验p值 = " r(N)/_N
双侧检验p值 = 0

. 


最后 `restore` 恢复原始数据，并删除中间文件 `bootstrap_twosample.dta`，保持工作目录整洁。


In [8]:
%%stata
restore
rm bootstrap_twosample.dta



. restore

. rm bootstrap_twosample.dta

. 
